In [ ]:
import json
import random
import time
from dataclasses import dataclass
from datetime import UTC, datetime
from typing import Iterator
import dataclasses

from kafka import KafkaProducer

In [ ]:
@dataclass(slots=True)
class VitalEvent:
    subject_id: str
    device_id: str
    event_time: str
    heart_rate: int
    oxygen_level: int
    systolic_bp: int
    diastolic_bp: int
    body_temperature: float
    activity_level: str
    alert_flag: bool

In [3]:
def _sample_activity_level() -> str:
    return random.choices(
        population=["sleeping", "resting", "walking", "running"],
        weights=[0.15, 0.45, 0.30, 0.10],
        k=1,
    )[0]


def _base_vitals(activity_level: str) -> tuple[int, int, int, int, float]:
    if activity_level == "sleeping":
        return (
            random.randint(55, 70),
            random.randint(95, 100),
            random.randint(95, 115),
            random.randint(60, 75),
            round(random.uniform(36.1, 36.8), 1),
        )
    if activity_level == "resting":
        return (
            random.randint(65, 85),
            random.randint(95, 100),
            random.randint(105, 125),
            random.randint(68, 82),
            round(random.uniform(36.3, 37.0), 1),
        )
    if activity_level == "walking":
        return (
            random.randint(80, 105),
            random.randint(94, 99),
            random.randint(110, 135),
            random.randint(70, 88),
            round(random.uniform(36.5, 37.3), 1),
        )
    return (
        random.randint(95, 135),
        random.randint(92, 98),
        random.randint(120, 150),
        random.randint(78, 95),
        round(random.uniform(36.8, 38.0), 1),
    )


def _inject_outlier(
    heart_rate: int,
    oxygen_level: int,
    systolic_bp: int,
    diastolic_bp: int,
    body_temperature: float,
) -> tuple[int, int, int, int, float]:
    if random.random() > 0.10:
        return heart_rate, oxygen_level, systolic_bp, diastolic_bp, body_temperature

    anomaly = random.choice(["tachycardia", "hypoxia", "hypertension", "fever"])
    if anomaly == "tachycardia":
        heart_rate = random.randint(121, 150)
    elif anomaly == "hypoxia":
        oxygen_level = random.randint(88, 91)
    elif anomaly == "hypertension":
        systolic_bp = random.randint(161, 185)
        diastolic_bp = random.randint(101, 120)
    else:
        body_temperature = round(random.uniform(38.1, 39.5), 1)

    return heart_rate, oxygen_level, systolic_bp, diastolic_bp, body_temperature


def _compute_alert_flag(event: VitalEvent) -> bool:
    return any(
        [
            event.heart_rate > 120,
            event.oxygen_level < 92,
            event.systolic_bp > 160,
            event.diastolic_bp > 100,
            event.body_temperature > 38.0,
        ]
    )


def generate_vital_event(subject_id: int, device_id: int) -> VitalEvent:
    activity_level = _sample_activity_level()
    heart_rate, oxygen_level, systolic_bp, diastolic_bp, body_temperature = _base_vitals(activity_level)
    heart_rate, oxygen_level, systolic_bp, diastolic_bp, body_temperature = _inject_outlier(
        heart_rate,
        oxygen_level,
        systolic_bp,
        diastolic_bp,
        body_temperature,
    )

    event = VitalEvent(
        subject_id=subject_id,
        device_id=device_id,
        event_time=datetime.now(UTC).isoformat(),
        heart_rate=heart_rate,
        oxygen_level=oxygen_level,
        systolic_bp=systolic_bp,
        diastolic_bp=diastolic_bp,
        body_temperature=body_temperature,
        activity_level=activity_level,
        alert_flag=False,
    )
    event.alert_flag = _compute_alert_flag(event)
    return event


def event_stream() -> Iterator[VitalEvent]:
    subject_ids = [f"{10000 + i}" for i in range(1, 40000)]
    device_ids = [f"{10000 + i}" for i in range(1, 26)]
    subject_device_map = dict(zip(subject_ids, random.choices(device_ids, k=len(subject_ids))))

    while True:
        subject_id = random.choice(subject_ids)
        device_id = subject_device_map[subject_id]
        yield generate_vital_event(subject_id, device_id)


In [4]:
sample_event = next(event_stream())
sample_event.to_dict()

{'subject_id': '36346',
 'device_id': '10014',
 'event_time': '2026-04-16T16:22:51.634632+00:00',
 'heart_rate': 80,
 'oxygen_level': 99,
 'systolic_bp': 127,
 'diastolic_bp': 86,
 'body_temperature': 37.0,
 'activity_level': 'walking',
 'alert_flag': False}

In [5]:
def event_serializer(event: VitalEvent) -> bytes:
    event_dict = dataclasses.asdict(event)
    event_json = json.dumps(event_dict).encode("utf-8")
    return event_json

In [6]:
server = "localhost:9092"

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=event_serializer,
)

In [7]:
topic_name = "healthcare-vitals"

def publish_event(event: VitalEvent) -> None:
    future = producer.send(
        topic=topic_name,
        value=event,
    )
    print(event.to_dict())
    metadata = future.get(timeout=10)
    print(
        {
            "topic": metadata.topic,
            "partition": metadata.partition,
            "offset": metadata.offset,
            "subject_id": event.subject_id,
            "alert_flag": event.alert_flag,
        }
    )


In [8]:
for event in event_stream():
    publish_event(event)
    time.sleep(1.0)

{'subject_id': '49977', 'device_id': '10021', 'event_time': '2026-04-16T16:22:51.689087+00:00', 'heart_rate': 100, 'oxygen_level': 95, 'systolic_bp': 130, 'diastolic_bp': 95, 'body_temperature': 36.9, 'activity_level': 'running', 'alert_flag': False}
{'topic': 'healthcare-vitals', 'partition': 0, 'offset': 106, 'subject_id': '49977', 'alert_flag': False}
{'subject_id': '27974', 'device_id': '10005', 'event_time': '2026-04-16T16:22:52.767233+00:00', 'heart_rate': 68, 'oxygen_level': 97, 'systolic_bp': 113, 'diastolic_bp': 76, 'body_temperature': 36.6, 'activity_level': 'resting', 'alert_flag': False}
{'topic': 'healthcare-vitals', 'partition': 0, 'offset': 107, 'subject_id': '27974', 'alert_flag': False}
{'subject_id': '26606', 'device_id': '10019', 'event_time': '2026-04-16T16:22:53.769204+00:00', 'heart_rate': 79, 'oxygen_level': 99, 'systolic_bp': 111, 'diastolic_bp': 71, 'body_temperature': 36.5, 'activity_level': 'resting', 'alert_flag': False}
{'topic': 'healthcare-vitals', 'parti

KeyboardInterrupt: 